# Texas StratMap 2025 Parcel Import → Supabase

**Goal:** Load all ~14 million Texas parcels into your `parcels_tx` table.

**Expected runtime:** 30-90 minutes (mostly unattended).

**Before running:**
1. Click the 🔑 key icon in the left sidebar (Colab Secrets)
2. Add a new secret named **`SUPABASE_DB_URL`**
3. Paste your direct Postgres connection string (the one starting with `postgresql://postgres:...`)
4. Toggle 'Notebook access' ON for the secret
5. Then click **Runtime → Run all**

**Resumability:** Already-imported counties are skipped automatically. If a cell fails mid-run, just click Run all again — it picks up where it stopped.

## 1. Install dependencies + connect to Supabase

In [ ]:
!pip install -q pyshp psycopg2-binary requests

import os, io, time, zipfile, tempfile, shutil, concurrent.futures
from pathlib import Path

import requests
import shapefile  # pyshp
import psycopg2

# Read the database URL from Colab Secrets (set in the left sidebar 🔑 panel)
try:
    from google.colab import userdata
    DB_URL = userdata.get('SUPABASE_DB_URL')
except Exception:
    DB_URL = os.environ.get('SUPABASE_DB_URL')

if not DB_URL:
    raise RuntimeError(
        '\n\nSUPABASE_DB_URL not set.\n'
        'Click the 🔑 Secrets icon in the left sidebar, add SUPABASE_DB_URL, paste your connection string, and toggle access ON.'
    )

# Test the connection
conn = psycopg2.connect(DB_URL)
with conn.cursor() as cur:
    cur.execute("SELECT current_database(), version()")
    db, ver = cur.fetchone()
    print(f'✓ Connected to: {db}')
    print(f'  Postgres:    {ver[:60]}...')
    cur.execute("SELECT COUNT(*)::int FROM parcels_tx")
    n = cur.fetchone()[0]
    print(f'  parcels_tx already has {n:,} rows')
conn.close()

## 2. Configuration

In [ ]:
# StratMap 2025 collection (TxGIO TNRIS DataHub)
COLLECTION_ID = '0fa04328-872e-481c-b453-126a74777593'
API_URL = f'https://api.tnris.org/api/v1/resources/?collection_id={COLLECTION_ID}'
DOWNLOAD_HEADERS = {
    'Referer': 'https://data.geographic.texas.gov/',
    'User-Agent': 'Mozilla/5.0',
}

# Where we store the downloaded zips on the Colab VM
ZIPS_DIR = Path('/content/parcels-2025/zips')
ZIPS_DIR.mkdir(parents=True, exist_ok=True)

# Shapefile DBF field name → parcels_tx column name (lowercase).
# These match what's in the StratMap schema (verified against Frio sample).
COLUMN_MAP = {
    'Prop_ID': 'prop_id', 'GEO_ID': 'geo_id', 'OWNER_NAME': 'owner_name',
    'NAME_CARE': 'name_care', 'LEGAL_AREA': 'legal_area', 'LGL_AREA_U': 'lgl_area_u',
    'GIS_AREA': 'gis_area', 'GIS_AREA_U': 'gis_area_u', 'LEGAL_DESC': 'legal_desc',
    'STAT_LAND_': 'stat_land_', 'LOC_LAND_U': 'loc_land_u', 'LAND_VALUE': 'land_value',
    'IMP_VALUE': 'imp_value', 'MKT_VALUE': 'mkt_value',
    'SITUS_ADDR': 'situs_addr', 'SITUS_NUM': 'situs_num', 'SITUS_STRE': 'situs_stre',
    'SITUS_ST_1': 'situs_st_1', 'SITUS_ST_2': 'situs_st_2', 'SITUS_CITY': 'situs_city',
    'SITUS_STAT': 'situs_stat', 'SITUS_ZIP': 'situs_zip',
    'MAIL_ADDR': 'mail_addr', 'MAIL_LINE1': 'mail_line1', 'MAIL_LINE2': 'mail_line2',
    'MAIL_CITY': 'mail_city', 'MAIL_STAT': 'mail_stat', 'MAIL_ZIP': 'mail_zip',
    'SOURCE': 'source', 'DATE_ACQ': 'date_acq', 'FIPS': 'fips',
    'COUNTY': 'county', 'TAX_YEAR': 'tax_year', 'YEAR_BUILT': 'year_built',
    'OBJECTID_1': 'objectid_1',
}
COPY_COLUMNS = list(COLUMN_MAP.values()) + ['geom']

print('Config loaded.')
print(f'  Will write {len(COPY_COLUMNS)} columns per row')
print(f'  Zip storage: {ZIPS_DIR}')

## 3. Download all 254 county zips (parallel, ~1-3 min)

In [ ]:
# Fetch the URL list from TNRIS API
print('Fetching URL list from TNRIS...')
r = requests.get(API_URL, timeout=30)
r.raise_for_status()
resources = r.json().get('results', [])
urls = [x['resource'] for x in resources if x['resource'].endswith('.zip')]
print(f'  Found {len(urls)} county zip URLs')

def download_one(url):
    fname = url.split('/')[-1]
    fpath = ZIPS_DIR / fname
    # Resume: skip if already downloaded AND has valid zip structure
    if fpath.exists() and fpath.stat().st_size > 1000:
        # Quick EOCD check
        with open(fpath, 'rb') as f:
            f.seek(max(0, fpath.stat().st_size - 65000))
            if b'PK\x05\x06' in f.read():
                return ('skip', fname, fpath.stat().st_size)
    # Download with retries
    try:
        with requests.get(url, headers=DOWNLOAD_HEADERS, stream=True, timeout=300) as resp:
            resp.raise_for_status()
            with open(fpath, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=1024*64):
                    if chunk: f.write(chunk)
        return ('ok', fname, fpath.stat().st_size)
    except Exception as e:
        return ('fail', fname, str(e))

t0 = time.time()
ok = skip = fail = 0
total_bytes = 0
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
    for i, (status, fname, info) in enumerate(pool.map(download_one, urls), 1):
        if status == 'ok':
            ok += 1
            total_bytes += info
        elif status == 'skip':
            skip += 1
            total_bytes += info
        else:
            fail += 1
            print(f'  ✗ {fname}: {info}')
        if i % 25 == 0:
            print(f'  ... {i}/{len(urls)} ({(time.time()-t0):.0f}s)')

print(f'\n✓ Downloads complete in {(time.time()-t0):.0f}s')
print(f'  {ok} downloaded, {skip} already present, {fail} failed')
print(f'  Total: {total_bytes / 1024**3:.2f} GB on disk')

## 4. Integrity check (skip corrupt zips)

In [ ]:
# Identify any corrupt zips (truncated download, etc.)
all_zips = sorted(ZIPS_DIR.glob('*.zip'))
bad_zips = []
for zp in all_zips:
    size = zp.stat().st_size
    if size < 1000:
        bad_zips.append(zp.name)
        continue
    with open(zp, 'rb') as f:
        f.seek(max(0, size - 65000))
        if b'PK\x05\x06' not in f.read():
            bad_zips.append(zp.name)

print(f'Total zips:    {len(all_zips)}')
print(f'Good zips:     {len(all_zips) - len(bad_zips)}')
print(f'Corrupt zips:  {len(bad_zips)}')
if bad_zips:
    print('\nCorrupt files (will be skipped — re-run cell 3 to retry):')
    for b in bad_zips[:20]:
        print(f'  ✗ {b}')
    # Move bad zips aside so the import loop skips them
    bad_dir = ZIPS_DIR.parent / 'bad'
    bad_dir.mkdir(exist_ok=True)
    for b in bad_zips:
        (ZIPS_DIR / b).rename(bad_dir / b)
    print(f'\n  Moved {len(bad_zips)} corrupt zips to {bad_dir}')

## 5. Import county by county (30-60 min)

In [ ]:
# Helpers ──────────────────────────────────────────────────────────────

def geometry_to_wkt(shape):
    """pyshp Shape → PostGIS WKT MultiPolygon (SRID=4326).
    Each ring becomes its own polygon in the multipolygon. For parcels
    with holes (rare), the hole gets filled — fine for autoLocate's
    point-in-polygon use case."""
    if shape is None or shape.shapeType not in (5, 15, 25):
        return None
    points = shape.points
    if not points:
        return None
    parts = list(shape.parts) + [len(points)]
    polygons = []
    for i in range(len(parts) - 1):
        ring = points[parts[i]:parts[i + 1]]
        if len(ring) < 4:
            continue
        # Close the ring if needed
        if ring[0] != ring[-1]:
            ring = list(ring) + [ring[0]]
        ring_str = ','.join(f'{x:.7f} {y:.7f}' for x, y in ring)
        polygons.append(f'(({ring_str}))')
    if not polygons:
        return None
    return f'SRID=4326;MULTIPOLYGON({",".join(polygons)})'

def escape_for_copy(v):
    """PostgreSQL COPY text format escaping."""
    if v is None:
        return r'\N'
    s = str(v)
    if not s.strip():
        return r'\N'
    return s.replace('\\', '\\\\').replace('\t', '\\t').replace('\n', '\\n').replace('\r', '\\r')

def import_county(conn, zip_path, force=False):
    """Import one county. Returns (status, rows, info) where status is
    'ok' | 'skip' | 'fail'."""
    fips = None
    for part in zip_path.stem.split('_'):
        if part.isdigit() and len(part) == 5:
            fips = part
            break
    if not fips:
        return ('fail', 0, 'no FIPS in filename')

    # Skip if already loaded (resume)
    with conn.cursor() as cur:
        cur.execute('SELECT COUNT(*)::int FROM parcels_tx WHERE fips = %s', (fips,))
        existing = cur.fetchone()[0]
    if existing > 0 and not force:
        return ('skip', existing, f'already has {existing:,} rows')
    if force and existing > 0:
        with conn.cursor() as cur:
            cur.execute('DELETE FROM parcels_tx WHERE fips = %s', (fips,))
        conn.commit()

    tmpdir = tempfile.mkdtemp(prefix='parcel-')
    try:
        # Extract
        try:
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(tmpdir)
        except zipfile.BadZipFile:
            return ('fail', 0, 'corrupt zip')

        # Find .shp
        shp_path = None
        for root, _, files in os.walk(os.path.join(tmpdir, 'shp')):
            for f in files:
                if f.endswith('.shp'):
                    shp_path = os.path.join(root, f)
                    break
            if shp_path: break
        if not shp_path:
            return ('fail', 0, 'no .shp in zip')

        # Read records + build COPY buffer
        sf = shapefile.Reader(shp_path, encoding='latin-1')
        field_names = [f[0] for f in sf.fields[1:]]  # skip DeletionFlag
        dbf_idx = {n: i for i, n in enumerate(field_names)}

        buf = io.StringIO()
        n_rows = 0
        n_skip = 0
        n_bad = 0
        total = len(sf)
        for i in range(total):
            try:
                rec = sf.record(i)
                shp = sf.shape(i)
            except Exception:
                n_bad += 1
                continue
            if shp is None or not getattr(shp, 'points', None):
                n_skip += 1
                continue
            try:
                wkt = geometry_to_wkt(shp)
            except Exception:
                n_bad += 1
                continue
            if wkt is None:
                n_skip += 1
                continue

            vals = []
            for src_name in COLUMN_MAP:
                v = rec[dbf_idx[src_name]] if src_name in dbf_idx else None
                vals.append(escape_for_copy(v))
            vals.append(escape_for_copy(wkt))
            buf.write('\t'.join(vals) + '\n')
            n_rows += 1

        if n_rows == 0:
            return ('fail', 0, f'no valid rows ({n_skip} empty, {n_bad} bad)')

        # COPY FROM STDIN — fastest path to Postgres
        buf.seek(0)
        try:
            with conn.cursor() as cur:
                cur.copy_from(buf, 'parcels_tx', columns=COPY_COLUMNS, sep='\t', null=r'\N')
            conn.commit()
        except Exception as e:
            conn.rollback()
            return ('fail', 0, f'COPY: {str(e)[:80]}')

        info = f'{n_rows:,} rows'
        if n_skip or n_bad:
            info += f' (+{n_skip} empty, {n_bad} bad)'
        return ('ok', n_rows, info)

    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

# Run the import ────────────────────────────────────────────────────────

conn = psycopg2.connect(DB_URL)
all_zips = sorted(ZIPS_DIR.glob('*.zip'))
print(f'Importing {len(all_zips)} county zips into parcels_tx...\n')

ok_count = skip_count = fail_count = 0
total_rows = 0
failed_zips = []
t0 = time.time()

for i, zp in enumerate(all_zips, 1):
    cstart = time.time()
    try:
        status, n, info = import_county(conn, zp)
    except Exception as e:
        try: conn.rollback()
        except: pass
        try:
            conn.close()
            conn = psycopg2.connect(DB_URL)
        except: pass
        status, n, info = 'fail', 0, f'uncaught: {str(e)[:80]}'
    elapsed = time.time() - cstart
    if status == 'ok':
        ok_count += 1
        total_rows += n
        print(f'[{i:>3}/{len(all_zips)}] {zp.name:55s} ✓ {info:<35} ({elapsed:.1f}s)')
    elif status == 'skip':
        skip_count += 1
        print(f'[{i:>3}/{len(all_zips)}] {zp.name:55s} ⏭  {info}')
    else:
        fail_count += 1
        failed_zips.append((zp.name, info))
        print(f'[{i:>3}/{len(all_zips)}] {zp.name:55s} ✗ {info}')

total_elapsed = time.time() - t0
print(f'\n{"═" * 75}')
print(f'  Import complete in {int(total_elapsed//60)}m {int(total_elapsed%60)}s')
print(f'  ✓ {ok_count} counties imported · {total_rows:,} rows')
print(f'  ⏭  {skip_count} skipped (already present)')
print(f'  ✗ {fail_count} failed')
print(f'{"═" * 75}')

if failed_zips:
    print('\nFailed counties (re-run cell 7 to retry):')
    for name, reason in failed_zips:
        print(f'  {name}: {reason}')

conn.close()

## 6. Verification — confirm search works against real data

In [ ]:
conn = psycopg2.connect(DB_URL)
with conn.cursor() as cur:
    cur.execute('SELECT COUNT(*)::bigint, COUNT(DISTINCT fips)::int FROM parcels_tx')
    total, counties = cur.fetchone()
    print(f'parcels_tx state:')
    print(f'  Total rows:       {total:,}')
    print(f'  Counties present: {counties} / 254')
    print()

    # Test the search function against known queries Christina's PDFs use
    test_queries = [
        ('Wesla Ranches', 'FRIO',       'Frio County / Wesla Farm'),
        ('Mikulencak',    'WILLIAMSON', 'Williamson / KWO Ranches grantee'),
        ('Bagan',         'GILLESPIE',  'Gillespie / Bagan grantor'),
        ('Fritz',         'FRIO',       'Frio / Fritz Farm'),
    ]
    print('Search function smoke tests:')
    for q, county, label in test_queries:
        cur.execute("SELECT jsonb_array_length((search_parcels_by_owner(%s, %s))->'features') AS hits", (q, county))
        hits = cur.fetchone()[0]
        ok = '✓' if hits > 0 else '✗'
        print(f'  {ok} {label:50s} → {hits} hits')
conn.close()

## 7. Retry failed counties (only run if cell 5 reported failures)

In [ ]:
# Only runs if cell 5 left some counties in `failed_zips`.
if not failed_zips:
    print('No failed counties to retry — skip this cell.')
else:
    print(f'Retrying {len(failed_zips)} failed counties...\n')
    conn = psycopg2.connect(DB_URL)
    new_failed = []
    for name, _ in failed_zips:
        zp = ZIPS_DIR / name
        if not zp.exists():
            print(f'  {name} — file missing, skip')
            continue
        try:
            status, n, info = import_county(conn, zp, force=True)
        except Exception as e:
            try: conn.rollback()
            except: pass
            status, n, info = 'fail', 0, f'uncaught: {e}'
        if status == 'ok':
            print(f'  ✓ {name}: {info}')
        else:
            print(f'  ✗ {name}: {info}')
            new_failed.append((name, info))
    conn.close()
    failed_zips = new_failed
    print(f'\nFinal: {len(failed_zips)} still failing')